## 1. 데이터 불러오기
> - 데이터는 Pandas 라이브러리의 pd.read_csv 함수를 활용하여 불러온다. 

In [ ]:
#필요 라이브러리 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas_profiling import ProfileReport

In [ ]:
#한글 폰트 지정
import matplotlib
import matplotlib.font_manager as fm

# fm.get_fontconfig_fonts()
font_location = 'C:/Windows/Fonts/malgun.ttf' # For Windows
font_name = fm.FontProperties(fname=font_location).get_name()
matplotlib.rc('font', family=font_name)

In [ ]:
#데이터 경로 지정
order_data_file_path = "./data/order_detail.csv"
customer_data_file_path = "./data/customer_data.csv"

In [ ]:
#데이터 불러오기
order_data = pd.read_csv(order_data_file_path)
customer_data = pd.read_csv(customer_data_file_path)

### Tip. 판다스 프로파일링 활용하기
> - (help) 판다스 프로파일링 설치 !pip install pandas_profiling

In [ ]:
#EDA 시작하기 (주문데이터)
ProfileReport(order_data)

In [ ]:
#EDA 시작하기 (고객데이터)
ProfileReport(customer_data)

## 2. 데이터 병합하기
> - 주문데이터와 고객데이터를 병합하기

In [ ]:
customer_data

In [ ]:
#주문데이터와 고객데이터 병합하기 (merge)
merge_data = pd.merge(order_data, customer_data, on="고객번호", how="left")

In [ ]:
#일자 데이터이기 떄문에, 연월일까지만 남기고 제거한다.
merge_data["가입일자"] = merge_data["가입일자"].str.slice(start=0, stop=10)

In [ ]:
#일자 데이터는 datetime 데이터 형식으로 변경한다.
merge_data["가입일자"] = pd.to_datetime(merge_data["가입일자"], format="%Y-%m-%d")

In [ ]:
#주문일자 컬럼을 생성한다.
merge_data["주문일자"] = merge_data["주문일시"].str.slice(start=0, stop=10)

In [ ]:
#일자 데이터는 datetime 데이터 형식으로 변경한다.
merge_data["주문일자"] = pd.to_datetime(merge_data["주문일자"], format="%Y-%m-%d")

In [ ]:
merge_data

## 3. 데이터 Divide 하기
> - 데이터 속의 데이터 찾기

In [ ]:
#브랜드/시즌/카테고리 정보 찾기
merge_data["브랜드명"] = merge_data["상품번호"].str.slice(3,5)
merge_data["시즌"] = merge_data["상품번호"].str.slice(5,9)
merge_data["카테고리"] = merge_data["상품번호"].str.slice(9,12)

In [ ]:
#브랜드별 구매 분포
brand_type = merge_data["브랜드명"].value_counts()

plt.bar(brand_type.index, brand_type.values, # 데이터의 x, 높이
        width = 0.8, bottom = None,        # Bar의 너비와 Bar 밑면의 y좌표
        align = 'edge')        # Bar의 정렬 세팅
plt.show()

In [ ]:
#시즌별 구매 분포
season_type = merge_data["시즌"].value_counts()

plt.bar(season_type.index, season_type.values, # 데이터의 x, 높이
        width = 0.8, bottom = None,        # Bar의 너비와 Bar 밑면의 y좌표
        align = 'edge')        # Bar의 정렬 세팅
plt.show()

In [ ]:
#시즌별 구매 분포
category_type = merge_data["카테고리"].value_counts()

plt.bar(category_type.index, category_type.values, # 데이터의 x, 높이
        width = 0.8, bottom = None,        # Bar의 너비와 Bar 밑면의 y좌표
        align = 'edge')        # Bar의 정렬 세팅
plt.show()

In [ ]:
# 연속형/이산형 데이터의 범주화 1
bins = list(range(10,100,10))
bins
bins_lable = [str(x)+"대" for x in bins]
bins_lable
merge_data["나이대"] = pd.cut(merge_data["나이"], bins, right=False, labels = bins_lable[:-1])

In [ ]:
merge_data["정상가"]

In [ ]:
# 연속형/이산형 데이터의 범주화 2
merge_data.loc[(merge_data["정상가"]>0)&(merge_data["정상가"]<=100000),"가격대"]="0만원~10만원대"
merge_data.loc[(merge_data["정상가"]>100000)&(merge_data["정상가"]<=200000),"가격대"]="10만원~20만원대"
merge_data.loc[(merge_data["정상가"]>200000)&(merge_data["정상가"]<=500000),"가격대"]="20만원~50만원대"
merge_data.loc[(merge_data["정상가"]>500000)&(merge_data["정상가"]<=1000000),"가격대"]="50만원~100만원대"
merge_data.loc[(merge_data["정상가"]>1000000),"가격대"]="100만원~"

## 4. 브랜드별/카테고리별 속성 알아보기

In [ ]:
brand_list = merge_data["브랜드명"].unique().tolist()
print(brand_list)

### Tip. Group by 함수를 활용하라

In [ ]:
pd.options.display.float_format = '{:.0f}'.format
merge_data.groupby("브랜드명").agg({"정상가":"mean","판매가":"sum","고객번호":"nunique","주문번호":"nunique"})

In [ ]:
pd.options.display.float_format = '{:.0f}'.format
merge_data.groupby("카테고리").agg({"정상가":"mean","판매가":"sum","고객번호":"nunique","주문번호":"nunique"})

In [ ]:
pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.max_rows', None)
merge_data.groupby(["브랜드명","카테고리"]).agg({"정상가":"mean","판매가":"sum","고객번호":"nunique","주문번호":"nunique"})

In [ ]:
merge_data[["브랜드명","판매가"]].boxplot(by="브랜드명")
plt.show()

## 5. 특정 브랜드의 판매가격 분포 확인

In [ ]:
ap_data = merge_data[merge_data["브랜드명"]=="AP"]
plt.title('AP 브랜드의 정상가격(TAG가) 분포')
plt.hist('정상가', bins = 50, range = (10000, 1000000), color='purple', data = ap_data)
plt.show()

In [ ]:
plt.title('AP 브랜드의 판매가격 분포')
plt.hist('판매가', bins = 50, range = (10000, 1000000), color='purple', data = ap_data)
plt.show()